In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from pathlib import Path

## ENV PATHS

In [ ]:
BASE_DIR_PATH = Path.cwd().parent
JDBC_DRIVER_PATH = f"{BASE_DIR_PATH}/drivers/postgresql-42.7.3.jar"
PG_HOST = ""
PG_PORT = 5432
PG_USER = ""
PG_PASSWORD = ""
PG_DATABASE = "vlr_events_metadata"
PG_TABLE = "agents"
jdbc_url = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
print(jdbc_url)

jdbc:postgresql://ep-misty-shadow-aipayev7-pooler.c-4.us-east-1.aws.neon.tech:5432/vlr_events_metadata


In [3]:
spark = (
    SparkSession.builder.master("local[*]")
    .config("spark.jars", JDBC_DRIVER_PATH)
    .appName("bronze-to-silver")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

26/03/05 17:17:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Agent Roles from External DB

Read the `agents` table from PostgreSQL.
We broadcast this later because it's a tiny lookup (~30 rows).

In [4]:
agent_roles_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", PG_TABLE)
    .option("user", PG_USER)
    .option("password", PG_PASSWORD)
    .option("driver", "org.postgresql.Driver")
    .load()
    # Normalize casing in lookup — ensures join works regardless of casing in DB
    .withColumn("agent", F.lower(F.trim(F.col("agent"))))
    .select("agent", "role")
)

print("agent_roles table from PostgreSQL:")
agent_roles_df.show(truncate=False)

agent_roles table from PostgreSQL:


+---------+----------+
|agent    |role      |
+---------+----------+
|jett     |Duelist   |
|reyna    |Duelist   |
|raze     |Duelist   |
|phoenix  |Duelist   |
|yoru     |Duelist   |
|neon     |Duelist   |
|iso      |Duelist   |
|sova     |Initiator |
|breach   |Initiator |
|skye     |Initiator |
|kayo     |Initiator |
|fade     |Initiator |
|gekko    |Initiator |
|brimstone|Controller|
|viper    |Controller|
|omen     |Controller|
|astra    |Controller|
|harbor   |Controller|
|clove    |Controller|
|killjoy  |Sentinel  |
+---------+----------+
only showing top 20 rows


## Bronze Schema

Schema must match the RAW CSV exactly — string columns stay as StringType
even if they'll become numeric later. We cast them explicitly after reading.

Partition columns (event_id, region, map, agent, snapshot_date) are included
because basePath injects them as real columns.

Note: `agents` column is excluded — it was a duplicate of the `agent`
partition column and has been dropped from Bronze.

In [5]:
BRONZE_SCHEMA = T.StructType([
    # --- Core player fields ---
    T.StructField("player_id",                    T.IntegerType(), nullable=False),
    T.StructField("player",                       T.StringType(),  nullable=False),
    T.StructField("org",                          T.StringType(),  nullable=True),
    T.StructField("agents",                          T.StringType(),  nullable=True),
    T.StructField("rounds_played",                T.IntegerType(), nullable=True),
    T.StructField("rating",                       T.DoubleType(),  nullable=True),
    T.StructField("average_combat_score",         T.DoubleType(),  nullable=True),
    # Renamed from kill_deaths → kill_death_ratio after read
    T.StructField("kill_deaths",                  T.DoubleType(),  nullable=True),
    # --- String fields that need parsing (keep as StringType in schema) ---
    T.StructField("kill_assists_survived_traded", T.StringType(),  nullable=True),  # "93%"
    T.StructField("average_damage_per_round",     T.DoubleType(),  nullable=True),
    T.StructField("kills_per_round",              T.StringType(),  nullable=True),  # "1.13"
    T.StructField("assists_per_round",            T.StringType(),  nullable=True),  # "0.53"
    T.StructField("first_kills_per_round",        T.StringType(),  nullable=True),  # "0.20"
    T.StructField("first_deaths_per_round",       T.DoubleType(),  nullable=True),
    T.StructField("headshot_percentage",          T.StringType(),  nullable=True),  # "28%"
    T.StructField("clutch_success_percentage",    T.StringType(),  nullable=True),  # "50%" or NULL
    T.StructField("clutches_won_played_ratio",    T.StringType(),  nullable=True),  # "1/2" or NULL
    T.StructField("max_kills_in_single_map",      T.IntegerType(), nullable=True),
    T.StructField("kills",                        T.IntegerType(), nullable=True),
    T.StructField("deaths",                       T.IntegerType(), nullable=True),
    T.StructField("assists",                      T.IntegerType(), nullable=True),
    T.StructField("first_kills",                  T.IntegerType(), nullable=True),
    T.StructField("first_deaths",                 T.IntegerType(), nullable=True),
    # --- Partition columns (injected by basePath) ---
    T.StructField("event_id",                     T.IntegerType(), nullable=False),
    T.StructField("region",                       T.StringType(),  nullable=False),
    T.StructField("map",                          T.StringType(),  nullable=False),
    T.StructField("agent",                        T.StringType(),  nullable=False),
    T.StructField("snapshot_date",                T.DateType(),    nullable=False),
])

## Read Bronze

In [6]:
BRONZE_DATA_PATH = BASE_DIR_PATH / "data" / "bronze"
SNAPSHOT_DATE = "2026-02-28"

snapshot_glob = (
    f"{BRONZE_DATA_PATH}/"
    f"event_id=*/region=*/map=*/agent=*/"
    f"snapshot_date={SNAPSHOT_DATE}/"
)

df = (
    spark.read
    .option("header", "true")
    .option("basePath", BRONZE_DATA_PATH)
    .schema(BRONZE_SCHEMA)
    .csv(snapshot_glob)
)

print(f"Rows loaded : {df.count():,}")
print(f"Columns     : {len(df.columns)}")
df.printSchema()

Rows loaded : 699,383
Columns     : 28
root
 |-- player_id: integer (nullable = true)
 |-- player: string (nullable = true)
 |-- org: string (nullable = true)
 |-- agents: string (nullable = true)
 |-- rounds_played: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- average_combat_score: double (nullable = true)
 |-- kill_deaths: double (nullable = true)
 |-- kill_assists_survived_traded: string (nullable = true)
 |-- average_damage_per_round: double (nullable = true)
 |-- kills_per_round: string (nullable = true)
 |-- assists_per_round: string (nullable = true)
 |-- first_kills_per_round: string (nullable = true)
 |-- first_deaths_per_round: double (nullable = true)
 |-- headshot_percentage: string (nullable = true)
 |-- clutch_success_percentage: string (nullable = true)
 |-- clutches_won_played_ratio: string (nullable = true)
 |-- max_kills_in_single_map: integer (nullable = true)
 |-- kills: integer (nullable = true)
 |-- deaths: integer (nullable = true)
 |-- as

In [7]:
# Inspect raw Bronze string columns before any transforms
df.select(
    "event_id", "player", "agent", "region", "map",
    "kill_assists_survived_traded",
    "kills_per_round",
    "headshot_percentage",
    "clutch_success_percentage",
    "clutches_won_played_ratio"
).show(10, truncate=False)

+--------+---------+-----+------+-----+----------------------------+---------------+-------------------+-------------------------+-------------------------+
|event_id|player   |agent|region|map  |kill_assists_survived_traded|kills_per_round|headshot_percentage|clutch_success_percentage|clutches_won_played_ratio|
+--------+---------+-----+------+-----+----------------------------+---------------+-------------------+-------------------------+-------------------------+
|60      |zekken   |sova |na    |haven|93%                         |1.13           |28%                |50%                      |1/2                      |
|60      |Kras     |sova |na    |haven|89%                         |1.28           |49%                |33%                      |1/3                      |
|60      |Cano     |sova |na    |haven|87%                         |1.30           |19%                |33%                      |3/9                      |
|60      |Lemonn   |sova |na    |haven|67%                

## Step 1 — Rename + String Normalization

In [8]:
# Rename kill_deaths → kill_death_ratio
df = df.withColumnRenamed("kill_deaths", "kill_death_ratio")

# Lowercase + trim categorical partition columns
for col in ["agent", "map", "region"]:
    df = df.withColumn(col, F.lower(F.trim(F.col(col))))

# Trim display columns only
for col in ["player", "org"]:
    df = df.withColumn(col, F.trim(F.col(col)))

print("After normalization:")
df.select("agent", "map", "region", "player", "org").show(5, truncate=False)

After normalization:
+-----+-----+------+-------+-----+
|agent|map  |region|player |org  |
+-----+-----+------+-------+-----+
|sova |haven|na    |zekken |MIBR |
|sova |haven|na    |Kras   |Mazov|
|sova |haven|na    |Cano   |Sho  |
|sova |haven|na    |Lemonn |IluZ |
|sova |haven|na    |Okeanos|EG   |
+-----+-----+------+-------+-----+
only showing top 5 rows


## Step 2 — Capture Raw Null Flags BEFORE Casting

**Why here?** Casting transforms nulls — `cast_percentage(NULL)` returns `NULL`,
but `cast_ratio_string("0/2")` returns `0.0` not null.
We must snapshot which rows had nulls in the RAW Bronze before any casting happens.

This is why `dq_clutch_no_attempts` was always 0 before —
the flag was set after casting had already filled/transformed the values.

In [9]:
# Capture raw null state from Bronze BEFORE any casting
# Both must be null → player was never in a clutch situation
# NULL percentage + "0/2" ratio → attempted clutches but won none (not flagged)
df = df.withColumn(
    "dq_clutch_no_attempts",
    F.col("clutch_success_percentage").isNull() &
    F.col("clutches_won_played_ratio").isNull()
)

# Verify — should match the 51,355 both-null rows we saw in raw Bronze
print(f"dq_clutch_no_attempts = True: {df.filter(F.col('dq_clutch_no_attempts')).count():,}")

dq_clutch_no_attempts = True: 51,355


## Step 3 — Type Casting

Three patterns from the raw data:

| Pattern | Example | Logic |
|---------|---------|-------|
| Percentage string | `"93%"` | strip `%`, cast Double, ÷ 100 |
| Plain decimal string | `"1.13"` | direct cast to Double |
| Ratio string | `"1/2"` | split `/` → numerator ÷ denominator |

In [10]:
def cast_percentage(col_name):
    """
    "93%" → 0.93
    Strips % symbol, casts to Double, divides by 100.
    NULL input → NULL output.
    """
    return (
        F.regexp_replace(F.col(col_name), "%", "")
         .cast(T.DoubleType())
         / F.lit(100.0)
    )


def cast_ratio_string(col_name):
    """
    "1/2" → 0.5  |  "0/2" → 0.0  |  NULL → NULL
    Splits on "/" and divides numerator by denominator.
    denom = 0 → NULL (guard against division by zero).
    """
    numerator   = F.split(F.col(col_name), "/").getItem(0).cast(T.DoubleType())
    denominator = F.split(F.col(col_name), "/").getItem(1).cast(T.DoubleType())

    return (
        F.when(F.col(col_name).isNull(), F.lit(None).cast(T.DoubleType()))
         .when(denominator == 0,         F.lit(None).cast(T.DoubleType()))
         .otherwise(F.round(numerator / denominator, 4))
    )


print("Helpers defined.")

Helpers defined.


In [11]:
# Percentage strings → 0.0-1.0 Double
df = df.withColumn("kill_assists_survived_traded", cast_percentage("kill_assists_survived_traded"))
df = df.withColumn("headshot_percentage",          cast_percentage("headshot_percentage"))
df = df.withColumn("clutch_success_percentage",    cast_percentage("clutch_success_percentage"))

# Plain decimal strings → Double
df = df.withColumn("kills_per_round",       F.col("kills_per_round").cast(T.DoubleType()))
df = df.withColumn("assists_per_round",     F.col("assists_per_round").cast(T.DoubleType()))
df = df.withColumn("first_kills_per_round", F.col("first_kills_per_round").cast(T.DoubleType()))

# Ratio string → Double
df = df.withColumn("clutches_won_played_ratio", cast_ratio_string("clutches_won_played_ratio"))

# Re-cast already-numeric columns — explicit guarantee regardless of Bronze quirks
df = df.withColumn("player_id",                F.col("player_id").cast(T.IntegerType()))
df = df.withColumn("rounds_played",            F.col("rounds_played").cast(T.IntegerType()))
df = df.withColumn("rating",                   F.col("rating").cast(T.DoubleType()))
df = df.withColumn("average_combat_score",     F.col("average_combat_score").cast(T.DoubleType()))
df = df.withColumn("kill_death_ratio",         F.col("kill_death_ratio").cast(T.DoubleType()))
df = df.withColumn("average_damage_per_round", F.col("average_damage_per_round").cast(T.DoubleType()))
df = df.withColumn("first_deaths_per_round",   F.col("first_deaths_per_round").cast(T.DoubleType()))
df = df.withColumn("max_kills_in_single_map",  F.col("max_kills_in_single_map").cast(T.IntegerType()))
df = df.withColumn("kills",                    F.col("kills").cast(T.IntegerType()))
df = df.withColumn("deaths",                   F.col("deaths").cast(T.IntegerType()))
df = df.withColumn("assists",                  F.col("assists").cast(T.IntegerType()))
df = df.withColumn("first_kills",              F.col("first_kills").cast(T.IntegerType()))
df = df.withColumn("first_deaths",             F.col("first_deaths").cast(T.IntegerType()))
df = df.withColumn("snapshot_date",            F.col("snapshot_date").cast(T.DateType()))

print("All casts applied.")
df.printSchema()

All casts applied.
root
 |-- player_id: integer (nullable = true)
 |-- player: string (nullable = true)
 |-- org: string (nullable = true)
 |-- agents: string (nullable = true)
 |-- rounds_played: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- average_combat_score: double (nullable = true)
 |-- kill_death_ratio: double (nullable = true)
 |-- kill_assists_survived_traded: double (nullable = true)
 |-- average_damage_per_round: double (nullable = true)
 |-- kills_per_round: double (nullable = true)
 |-- assists_per_round: double (nullable = true)
 |-- first_kills_per_round: double (nullable = true)
 |-- first_deaths_per_round: double (nullable = true)
 |-- headshot_percentage: double (nullable = true)
 |-- clutch_success_percentage: double (nullable = true)
 |-- clutches_won_played_ratio: double (nullable = true)
 |-- max_kills_in_single_map: integer (nullable = true)
 |-- kills: integer (nullable = true)
 |-- deaths: integer (nullable = true)
 |-- assists: integer 

In [12]:
# Validate cast results — spot check values are in expected ranges
df.select(
    "kill_assists_survived_traded",  # expect 0.0 - 1.0
    "headshot_percentage",           # expect 0.0 - 1.0
    "clutch_success_percentage",     # expect 0.0 - 1.0 or NULL
    "clutches_won_played_ratio",     # expect 0.0 - 1.0 or NULL
    "kills_per_round",               # expect 0.3 - 1.5 range typically
).show(10)

+----------------------------+-------------------+-------------------------+-------------------------+---------------+
|kill_assists_survived_traded|headshot_percentage|clutch_success_percentage|clutches_won_played_ratio|kills_per_round|
+----------------------------+-------------------+-------------------------+-------------------------+---------------+
|                        0.93|               0.28|                      0.5|                      0.5|           1.13|
|                        0.89|               0.49|                     0.33|                   0.3333|           1.28|
|                        0.87|               0.19|                     0.33|                   0.3333|            1.3|
|                        0.67|               0.41|                     0.13|                    0.125|           1.16|
|                        0.87|               0.46|                      1.0|                      1.0|            1.0|
|                        0.82|               0.3

## Step 4 — Remaining DQ Flags + Clutch Null Fill

`dq_clutch_no_attempts` was already set in Step 2 (before casting).
All other flags run here after casting so comparisons are on numeric values.

In [13]:
RATIO_TOLERANCE = 0.05  # 5%


def ratio_mismatch(stored_col, numerator_col, denominator_col):
    """
    True if stored VLR ratio deviates > RATIO_TOLERANCE from raw totals.
    Two nullif guards:
      1. denominator → prevents kills/deaths divide by zero
      2. safe_computed → prevents deviation/computed divide by zero
         when computed itself is 0 (e.g. 0 kills / 10 rounds = 0.0)
    """
    denom         = F.nullif(F.col(denominator_col), F.lit(0))
    computed      = F.col(numerator_col) / denom
    stored        = F.col(stored_col)
    safe_computed = F.nullif(computed, F.lit(0.0))
    deviation     = F.abs(stored - safe_computed) / F.abs(safe_computed)
    return (
        stored.isNotNull()        &
        safe_computed.isNotNull() &
        (deviation > RATIO_TOLERANCE)
    )


# dq_low_sample
df = df.withColumn(
    "dq_low_sample",
    F.col("rounds_played").isNull() | (F.col("rounds_played") < 50)
)

# Fill clutch nulls with 0.0 AFTER dq_clutch_no_attempts was already captured
df = df.withColumn(
    "clutch_success_percentage",
    F.coalesce(F.col("clutch_success_percentage"), F.lit(0.0))
)
df = df.withColumn(
    "clutches_won_played_ratio",
    F.coalesce(F.col("clutches_won_played_ratio"), F.lit(0.0))
)

# dq_null_core_fields
df = df.withColumn(
    "dq_null_core_fields",
    F.col("rating").isNull() |
    F.col("average_combat_score").isNull() |
    F.col("kills").isNull()
)

# dq_ratio_mismatch
df = df.withColumn(
    "dq_ratio_mismatch",
    ratio_mismatch("kill_death_ratio",       "kills",        "deaths")        |
    ratio_mismatch("kills_per_round",         "kills",        "rounds_played") |
    ratio_mismatch("first_kills_per_round",   "first_kills",  "rounds_played") |
    ratio_mismatch("first_deaths_per_round",  "first_deaths", "rounds_played")
)

print("DQ flags added.")

DQ flags added.


In [14]:
# DQ Summary
total = df.count()
dq_cols = [
    "dq_low_sample",
    "dq_clutch_no_attempts",
    "dq_null_core_fields",
    "dq_ratio_mismatch",
]

print(f"Total rows: {total:,}")
print("-" * 45)
for flag in dq_cols:
    count = df.filter(F.col(flag) == True).count()
    pct   = (count / total * 100) if total > 0 else 0
    print(f"  {flag:<28} {count:>6,}  ({pct:.1f}%)")

Total rows: 699,383
---------------------------------------------


  dq_low_sample                496,077  (70.9%)


  dq_clutch_no_attempts        51,355  (7.3%)


  dq_null_core_fields          46,351  (6.6%)


  dq_ratio_mismatch            203,664  (29.1%)


In [15]:
# Inspect ratio mismatch rows — understand WHY they're flagged
# before deciding if the flag is meaningful or if tolerance needs adjusting
df.filter(F.col("dq_ratio_mismatch") == True).select(
    "player",
    "kills", "deaths",
    "kill_death_ratio",
    F.round(F.col("kills") / F.nullif(F.col("deaths"), F.lit(0)), 3).alias("computed_kd"),
    "kills", "rounds_played",
    "kills_per_round",
    F.round(F.col("kills") / F.nullif(F.col("rounds_played"), F.lit(0)), 3).alias("computed_kpr"),
).show(20, truncate=False)

+----------------+-----+------+----------------+-----------+-----+-------------+---------------+------------+
|player          |kills|deaths|kill_death_ratio|computed_kd|kills|rounds_played|kills_per_round|computed_kpr|
+----------------+-----+------+----------------+-----------+-----+-------------+---------------+------------+
|Cano            |39   |21    |1.86            |1.857      |39   |30           |1.3            |1.3         |
|Lemonn          |52   |25    |2.08            |2.08       |52   |45           |1.16           |1.156       |
|Tsuya           |37   |21    |1.76            |1.762      |37   |37           |1.0            |1.0         |
|ryder           |25   |17    |1.47            |1.471      |25   |22           |1.14           |1.136       |
|devie           |23   |15    |1.53            |1.533      |23   |23           |1.0            |1.0         |
|Jvnko           |18   |14    |1.29            |1.286      |18   |18           |1.0            |1.0         |
|Notlast  

## Step 5 — Derived Metrics

Computed from raw totals — not VLR's pre-computed ratios.

| Column | Formula | Meaning |
|--------|---------|--------|
| `fk_fd_ratio` | `first_kills / first_deaths` | Entry fragger efficiency. >1 = winning opening duels |
| `net_first_blood` | `first_kills - first_deaths` | Net opening duel impact per event context |
| `damage_delta` | `adr - 150` | ADR above/below one kill's worth of damage per round |

In [16]:
# fk_fd_ratio
# first_deaths=0, first_kills>0 → perfect entry record → sentinel 99.0
# first_deaths=0, first_kills=0 → never entry fragged → NULL
df = df.withColumn(
    "fk_fd_ratio",
    F.when(
        F.col("first_deaths").isNull() | F.col("first_kills").isNull(),
        F.lit(None).cast(T.DoubleType())
    ).when(
        (F.col("first_deaths") == 0) & (F.col("first_kills") > 0),
        F.lit(99.0)
    ).when(
        (F.col("first_deaths") == 0) & (F.col("first_kills") == 0),
        F.lit(None).cast(T.DoubleType())
    ).otherwise(
        F.round(F.col("first_kills") / F.col("first_deaths"), 3)
    )
)

# net_first_blood
df = df.withColumn(
    "net_first_blood",
    F.when(
        F.col("first_kills").isNotNull() & F.col("first_deaths").isNotNull(),
        F.col("first_kills") - F.col("first_deaths")
    ).otherwise(F.lit(None).cast(T.IntegerType()))
)

# damage_delta
df = df.withColumn(
    "damage_delta",
    F.when(
        F.col("average_damage_per_round").isNotNull(),
        F.round(F.col("average_damage_per_round") - 150.0, 2)
    ).otherwise(F.lit(None).cast(T.DoubleType()))
)

# Spot check
df.select(
    "player", "first_kills", "first_deaths",
    "fk_fd_ratio", "net_first_blood",
    "average_damage_per_round", "damage_delta"
).show(10)

+---------+-----------+------------+-----------+---------------+------------------------+------------+
|   player|first_kills|first_deaths|fk_fd_ratio|net_first_blood|average_damage_per_round|damage_delta|
+---------+-----------+------------+-----------+---------------+------------------------+------------+
|   zekken|          3|           0|       99.0|              3|                   238.8|        88.8|
|     Kras|          3|           0|       99.0|              3|                   227.6|        77.6|
|     Cano|          2|           2|        1.0|              0|                   228.3|        78.3|
|   Lemonn|          3|           2|        1.5|              1|                   215.0|        65.0|
|  Okeanos|          2|           2|        1.0|              0|                   181.1|        31.1|
|     sogo|          3|           0|       99.0|              3|                   171.3|        21.3|
|    Tsuya|          3|           2|        1.5|              1|         

## Step 6 — Agent Role Enrichment

Broadcast join the PostgreSQL lookup onto the Silver DataFrame.
Left join so unknown agents keep their rows with `role = null`.

In [17]:
df = df.join(F.broadcast(agent_roles_df), on="agent", how="left")

# Check for agents not in the lookup
unknown = df.filter(F.col("role").isNull()).select("agent").distinct().collect()

if unknown:
    print(f"WARNING — agents not in lookup: {sorted([r['agent'] for r in unknown])}")
    print("Add them to the agents table in PostgreSQL.")
else:
    print("All agents resolved to a role. ✓")

df.select("agent", "role").distinct().orderBy("role", "agent").show(30, truncate=False)

All agents resolved to a role. ✓


+---------+----------+
|agent    |role      |
+---------+----------+
|astra    |Controller|
|brimstone|Controller|
|clove    |Controller|
|harbor   |Controller|
|omen     |Controller|
|viper    |Controller|
|iso      |Duelist   |
|jett     |Duelist   |
|neon     |Duelist   |
|phoenix  |Duelist   |
|raze     |Duelist   |
|reyna    |Duelist   |
|waylay   |Duelist   |
|yoru     |Duelist   |
|breach   |Initiator |
|fade     |Initiator |
|gekko    |Initiator |
|kayo     |Initiator |
|skye     |Initiator |
|sova     |Initiator |
|tejo     |Initiator |
|chamber  |Sentinel  |
|cypher   |Sentinel  |
|deadlock |Sentinel  |
|killjoy  |Sentinel  |
|sage     |Sentinel  |
|veto     |Sentinel  |
|vyse     |Sentinel  |
+---------+----------+



## Step 7 — Final Column Order + Schema Review

In [18]:
# Identity → Context → Volume → Core metrics → Derived → Raw totals → DQ flags
SILVER_COLUMNS = [
    # Identity
    "player_id", "player", "org",
    # Context
    "event_id", "region", "map", "agent", "role", "snapshot_date",
    # Volume
    "rounds_played",
    # Core metrics (cleaned)
    "rating", "average_combat_score", "kill_death_ratio",
    "kill_assists_survived_traded", "average_damage_per_round",
    "kills_per_round", "assists_per_round",
    "first_kills_per_round", "first_deaths_per_round",
    "headshot_percentage", "clutch_success_percentage",
    "clutches_won_played_ratio", "max_kills_in_single_map",
    # Derived metrics
    "fk_fd_ratio", "net_first_blood", "damage_delta",
    # Raw totals
    "kills", "deaths", "assists", "first_kills", "first_deaths",
    # DQ flags
    "dq_low_sample", "dq_clutch_no_attempts",
    "dq_null_core_fields", "dq_ratio_mismatch",
]

df = df.select(SILVER_COLUMNS)

print(f"Final Silver rows    : {df.count():,}")
print(f"Final Silver columns : {len(df.columns)}")
df.printSchema()

Final Silver rows    : 699,383
Final Silver columns : 35
root
 |-- player_id: integer (nullable = true)
 |-- player: string (nullable = true)
 |-- org: string (nullable = true)
 |-- event_id: integer (nullable = true)
 |-- region: string (nullable = true)
 |-- map: string (nullable = true)
 |-- agent: string (nullable = true)
 |-- role: string (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- rounds_played: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- average_combat_score: double (nullable = true)
 |-- kill_death_ratio: double (nullable = true)
 |-- kill_assists_survived_traded: double (nullable = true)
 |-- average_damage_per_round: double (nullable = true)
 |-- kills_per_round: double (nullable = true)
 |-- assists_per_round: double (nullable = true)
 |-- first_kills_per_round: double (nullable = true)
 |-- first_deaths_per_round: double (nullable = true)
 |-- headshot_percentage: double (nullable = true)
 |-- clutch_success_percentage: double 

## Step 8 — Write Silver (Parquet)

Partitioned by `event_id` and `region`.
Fewer partitions than Bronze — `snapshot_date`, `map`, `agent` are now
regular columns, not folder partitions.

In [19]:
SILVER_PATH = BASE_DIR_PATH / "data" / "silver"

(
    df.write
    .format("parquet")
    .mode("overwrite")
    .partitionBy("event_id", "region")
    .save(str(SILVER_PATH))
)

print(f"Silver written to: {SILVER_PATH}")

Silver written to: /Users/pvcodes/Data/Codes/dev/vct-analytics/functions/vlr-silver-transform/data/silver


In [20]:
# Read back and verify
df_verify = spark.read.parquet(str(SILVER_PATH))

print(f"Rows written   : {df.count():,}")
print(f"Rows read back : {df_verify.count():,}")
print()
print("Partitions written:")
import os
for entry in sorted(os.listdir(SILVER_PATH)):
    if not entry.startswith("."):
        print(f"  {entry}")

Rows written   : 699,383


Rows read back : 699,383

Partitions written:
  _SUCCESS
  event_id=1
  event_id=10
  event_id=11
  event_id=12
  event_id=13
  event_id=14
  event_id=15
  event_id=17
  event_id=19
  event_id=2
  event_id=20
  event_id=21
  event_id=22
  event_id=23
  event_id=24
  event_id=25
  event_id=26
  event_id=27
  event_id=28
  event_id=29
  event_id=3
  event_id=30
  event_id=31
  event_id=32
  event_id=33
  event_id=34
  event_id=35
  event_id=36
  event_id=37
  event_id=38
  event_id=39
  event_id=40
  event_id=41
  event_id=42
  event_id=43
  event_id=44
  event_id=45
  event_id=46
  event_id=47
  event_id=48
  event_id=49
  event_id=5
  event_id=50
  event_id=51
  event_id=54
  event_id=55
  event_id=56
  event_id=57
  event_id=58
  event_id=59
  event_id=6
  event_id=60
  event_id=61
  event_id=62
  event_id=63
  event_id=64
  event_id=65
  event_id=66
  event_id=67
  event_id=69
  event_id=7
  event_id=70
  event_id=72
  event_id=73
  event_id=74
  event_id=75
  event_id=76
  event_id=

In [21]:
spark.stop()
print("Spark session stopped. Silver pipeline complete.")

Spark session stopped. Silver pipeline complete.
